In [19]:
from pathlib import Path

import pandas as pd

In [20]:
SOURCE_DIR = Path("src")
OUTPUT_DIR = Path("out")
OUTPUT_DIR.mkdir(exist_ok=True)

BY_ARTICLE_PATH = SOURCE_DIR / "by_article.csv"
YEAR_TOTALS_PATH = SOURCE_DIR / "year_totals.csv"

for path in [BY_ARTICLE_PATH, YEAR_TOTALS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")

In [21]:
by_article = pd.read_csv(BY_ARTICLE_PATH)
year_totals = pd.read_csv(YEAR_TOTALS_PATH)

by_article["Wert_Anzahl"] = pd.to_numeric(by_article["Wert_Anzahl"], errors="coerce")
year_totals["Wert_Anzahl"] = pd.to_numeric(year_totals["Wert_Anzahl"], errors="coerce")

In [22]:
year_summary = (
    year_totals.pivot_table(
        index="source_year",
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
    .sort_values("source_year")
)

year_summary["verurteilungsquote"] = year_summary["verurteilte"] / year_summary["abgeurteilte"]
year_summary["verurteilungsquote_pct"] = year_summary["verurteilungsquote"] * 100
year_summary[["abgeurteilte", "verurteilte"]] = year_summary[["abgeurteilte", "verurteilte"]].astype(int)

year_summary

Personengruppe,source_year,abgeurteilte,verurteilte,verurteilungsquote,verurteilungsquote_pct
0,2022,791090,647374,0.818332,81.833167
1,2023,804410,656901,0.816625,81.662461
2,2024,781610,632115,0.808735,80.873454


In [23]:
article_base = by_article.loc[
    (by_article["Geschlecht"] == "Insgesamt")
    & (by_article["Altersgruppe"] == ".")
    & (by_article["Angewandtes_Strafrecht"] == ".")
    & (by_article["Personengruppe"].isin(["Abgeurteilte insgesamt", "Verurteilte insgesamt"]))
].copy()

article_summary = (
    article_base.pivot_table(
        index=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
)

article_summary = article_summary.dropna(subset=["abgeurteilte", "verurteilte"])
article_summary = article_summary.loc[article_summary["abgeurteilte"] > 0].copy()
article_summary[["abgeurteilte", "verurteilte"]] = article_summary[["abgeurteilte", "verurteilte"]].astype(int)
article_summary["verurteilungsquote"] = article_summary["verurteilte"] / article_summary["abgeurteilte"]
article_summary["verurteilungsquote_pct"] = article_summary["verurteilungsquote"] * 100

article_summary.sort_values(["source_year", "abgeurteilte"], ascending=[True, False]).head(20)

Personengruppe,source_year,Straftat_Gesetz,Art_der_Straftat,abgeurteilte,verurteilte,verurteilungsquote,verurteilungsquote_pct
464,2022,StGBoV,Straftaten ohne Straftaten im Straßenverkehr n...,475914,370989,0.779529,77.952949
477,2022,Straftat,Straftaten außerhalb des Strafgesetzbuches (St...,191045,167355,0.875998,87.599780
433,2022,StGB,"StGB §§ 142, 315 b bis d,316 sowie 222,229, §§...",185490,164153,0.884970,88.496954
12,2022,And BuG,Straftaten nach anderen Bundes- und Landesgese...,129686,112232,0.865413,86.541338
109,2022,StGB,"StGB 22. Abschnitt, §§ 263 bis 266 b Betrug un...",124664,103392,0.829365,82.936533
455,2022,StGB,Straftaten im Straßenverkehr nach dem StGB ins...,124131,109030,0.878346,87.834626
456,2022,StGB,Straftaten im Straßenverkehr nach dem StGB und...,109244,91355,0.836247,83.624730
105,2022,StGB,"StGB 19. Abschnitt, §§ 242 bis 248 c Diebstahl...",106435,86164,0.809546,80.954573
264,2022,StGB,StGB § 242 Diebstahl,79688,65244,0.818743,81.874310
454,2022,StGB,Straftaten im Straßenverkehr nach dem StGB in ...,76246,72798,0.954778,95.477796


In [24]:
YEAR_SUMMARY_OUTPUT = OUTPUT_DIR / "year_conviction_rates.csv"
ARTICLE_SUMMARY_OUTPUT = OUTPUT_DIR / "article_conviction_rates.csv"

year_summary.to_csv(YEAR_SUMMARY_OUTPUT, index=False)
article_summary.to_csv(ARTICLE_SUMMARY_OUTPUT, index=False)

print(f"Wrote {YEAR_SUMMARY_OUTPUT} ({len(year_summary)} rows)")
print(f"Wrote {ARTICLE_SUMMARY_OUTPUT} ({len(article_summary)} rows)")

Wrote out\year_conviction_rates.csv (3 rows)
Wrote out\article_conviction_rates.csv (1471 rows)


In [ ]:
stgb_article_summary = article_summary.loc[
    article_summary["Straftat_Gesetz"].astype(str).str.strip().isin(["StGB", "StGBoV"])
].copy()

stgb_year_base = article_summary.loc[
    (
        (article_summary["Straftat_Gesetz"].astype(str).str.strip() == "StGBoV")
        & article_summary["Art_der_Straftat"].astype(str).str.contains(
            "Straftaten ohne Straftaten im Straßenverkehr nach dem StGB insgesamt Summe",
            na=False,
        )
    )
    |
    (
        (article_summary["Straftat_Gesetz"].astype(str).str.strip() == "StGB")
        & article_summary["Art_der_Straftat"].astype(str).str.contains(
            "Straftaten im Straßenverkehr nach dem StGB insgesamt Summe",
            na=False,
        )
    )
].copy()

stgb_year_summary = stgb_year_base.groupby("source_year", as_index=False)[["abgeurteilte", "verurteilte"]].sum().sort_values("source_year")
stgb_year_summary["verurteilungsquote"] = stgb_year_summary["verurteilte"] / stgb_year_summary["abgeurteilte"]
stgb_year_summary["verurteilungsquote_pct"] = stgb_year_summary["verurteilungsquote"] * 100

stgb_year_summary

In [ ]:
STGB_YEAR_OUTPUT = OUTPUT_DIR / "stgb_year_conviction_rates.csv"
STGB_ARTICLE_OUTPUT = OUTPUT_DIR / "stgb_article_conviction_rates.csv"

stgb_year_summary.to_csv(STGB_YEAR_OUTPUT, index=False)
stgb_article_summary.to_csv(STGB_ARTICLE_OUTPUT, index=False)

print(f"Wrote {STGB_YEAR_OUTPUT} ({len(stgb_year_summary)} rows)")
print(f"Wrote {STGB_ARTICLE_OUTPUT} ({len(stgb_article_summary)} rows)")